Use this after running the script `psdata_conversion_and_consolidation.ipynb`

Step 0: load libraries and specify the data file path

In [ ]:
import os
import gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.covariance import EllipticEnvelope
from sklearn.preprocessing import MaxAbsScaler
from scipy.signal import savgol_filter
from scipy.signal import find_peaks
from scipy.optimize import differential_evolution

In [ ]:
# Specify the top folder
top_folder = r'N:\FWET\FDCH\AdsCatal\General\personal_work_folders\plasmacatdesign\co2-splitting\uhasselt\SiO2+TMAH-220-12H\pwr-const\lissajous\55.0s-02-plasma-01'

# Column names for the result .csv file
column_names_new = [
	'project_name',
	'psdata_file_name',
	'measurement_date',
	'measurement_number',
	'material_supplier',
	'material_name',
	'reaction_type',
	'wattage_const',
	'residence_time_s',
	'plasma_state',
	'cycle',
	'power_plasma_W',
	'power_plasma_UIprod_W',
	'power_source_W',
	'U_pp_V',
	'current_rms_reactor_A',
	'current_rms_plasma_A',
	'current_rms_source_A',
	'U_burning_neg_V',
	'U_burning_pos_V',
	'U_burning_avg_V',
	'U_breakdown_neg_V',
	'U_breakdown_pos_V',
	'U_breakdown_avg_V',
	'Q_delta_dis_pos_C',
	'Q_delta_dis_neg_C',
	'Q_delta_dis_avg_C',
	'avg_num_udisch_per_cycle',
	'C_cell_neg_F',
	'C_cell_neg_intrcpt_C',
	'C_cell_pos_F',
	'C_cell_pos_intrcpt_C',
	'C_cell_avg_F',
	'C_diel_eff_neg_F',
	'C_diel_eff_neg_intrcpt_C',
	'C_diel_eff_pos_F',
	'C_diel_eff_pos_intrcpt_C',
	'C_diel_eff_avg_F',
	'alpha_neg',
	'alpha_pos',
	'alpha_avg',
	'beta_neg',
	'beta_pos',
	'beta_avg'
]

In [ ]:
def lissajous_area(x, y):
	# Perform numerical integration of the Lissaous curve
	return 0.5 * np.abs(np.dot(x, np.roll(y, 1)) - np.dot(y, np.roll(x, 1)))

In [ ]:
def objective_function(
        shift, data, U, Q, ac_freq_Hz, num_cycles, power_plasma_UIprod_W
):
    # Round the shift to the nearest integer
    shift = int(np.round(shift).item())
    
    # Apply the shift to the charge column
    shifted_charge = np.roll(data[Q], -shift)
    
    # Calculate plasma power using the Lissajous area
    power_plasma_W = lissajous_area(data[U], shifted_charge) * ac_freq_Hz / num_cycles
    
    # Return the absolute difference between the two power values
    return np.abs(power_plasma_W - power_plasma_UIprod_W)

To maintain a better overview, let us define multiple functions each responsible of a specific task:
- `calculate_results` calculates the results for the given DataFrame and returns a new DataFrame with the results and writes those results to a .parquet file
- `remove_outliers` removes outliers from the generated DatFrame in the previous step and writes those results to a second .parquet file
- `lissajous_generation` generates a Lissajous figure from the given DataFrame by averaging
- `current_voltage_profile` generates a .parquet file with the current and voltage profile of the given DataFrame but reduced in size

Step 1: `calculate_results`

Define calculate_results function that calculates the results for the given DataFrame and returns a new DataFrame with the results

In [ ]:
def calculate_results(
		data_df,
		result_column_names,
		ac_freq_Hz=3000,
		C_diel_F=204e-12,
		C_gas_F=7.4e-12,
		discharge_threshold_A=7e-3,
		t='time_s',
		Q='charge_C',
		U='voltage_V',
		Ir='current_reactor_A',
		Is='current_source_A',
		smoothing_window_size=15625
):
	# Extract some variables from the data
	project_name = data_df['project_name'].iloc[0]
	reaction_type = data_df['reaction_type'].iloc[0]
	material_supplier = data_df['material_supplier'].iloc[0]
	material_name = data_df['material_name'].iloc[0]
	wattage_const = data_df['wattage_const'].iloc[0]
	residence_time_s = data_df['residence_time_s'].iloc[0]
	measurement_number = data_df['measurement_number'].iloc[0]
	plasma_state = data_df['plasma_state'].iloc[0]
	date = data_df['date'].iloc[0]

	# Other variables
	ac_period_s = 1 / ac_freq_Hz

	# Initialize the result DataFrame
	results_psdata_files = []

	# Initialize columns for smoothed data in the original dataframe
	data_df['dV_dt'] = np.nan
	data_df['dQ_dV'] = np.nan

	# Loop over the grouped data by the column 'psdata_file_name'
	for psdata_file_name, group_data in data_df.groupby('psdata_file_name'):
		print(f'Processing {psdata_file_name}...')

		data = group_data.copy()

		# Sort data by 't'
		data.sort_values(by=t, inplace=True)

		# Make t start at 0
		data[t] = data[t] - data[t].iloc[0]

		# Check sign of 1st value in current_reactor_A/Ir column
		if np.sign(data[Ir].iloc[0]) == -1:
			# Flip the sign of all values in the column if negative
			data[Ir] = -data[Ir]

		# Calculate the number of full cycles
		num_cycles = int(np.floor(
			(data[t].iloc[-1] - data[t].iloc[0]) / ac_period_s
		))

		# Calculate the total time of the full cycles
		full_cycle_time = num_cycles * ac_period_s

		# Select the full cycles
		data = data[data[t] < data[t].iloc[0] + full_cycle_time]

		# Calculate plasma power using the product of U and I
		power_plasma_UIprod_W = np.mean(data[U] * data[Ir])

		# Calculate time per row
		t_per_row = data[t].iloc[1] - data[t].iloc[0]

		# Define the bounds for the shift
		shift_bounds = [(0, int((ac_period_s / 10) / t_per_row))]

		# Run the differential evolution algorithm
		result = differential_evolution(
			objective_function,
			bounds=shift_bounds,
			args=(data, U, Q, ac_freq_Hz, num_cycles, power_plasma_UIprod_W),
			strategy='best1bin',
			maxiter=1000,
			popsize=15,
			tol=1e-6,
			mutation=(0.5, 1.0),
			recombination=0.7
		)

		# Extract the optimized shift value
		optimized_shift = int(np.round(result.x[0]))

		print(f"Optimized shift: {optimized_shift * t_per_row * 1e6} µs")

		# Apply the optimized shift to the charge column
		data.loc[:, Q] = np.roll(data[Q], -optimized_shift)

		# Smooth voltage_V and charge_C using a Savitzky-Golay filter
		data.loc[:, 'voltage_smooth_V'] = savgol_filter(
			data[U],
			window_length=smoothing_window_size,
			polyorder=2
		)

		data.loc[:, 'charge_smooth_C'] = savgol_filter(
			data[Q],
			window_length=smoothing_window_size,
			polyorder=2
		)

		# Calculate necessary derivatives
		data.loc[:, 'dV_dt'] = np.gradient(
					data['voltage_smooth_V'],
					data[t]
				)
		
		data.loc[:, 'dQ_dV'] = np.gradient(
			data['charge_smooth_C'],
			data['voltage_smooth_V']
		)

		# Update the original dataframe with
		# corrected time, shifted charge, and smoothed values
		data_df.loc[data.index, t] = data[t]
		data_df.loc[data.index, Q] = data[Q]
		data_df.loc[data.index, 'dV_dt'] = data['dV_dt']
		data_df.loc[data.index, 'dQ_dV'] = data['dQ_dV']

		# Initialize the result dictionary
		results_cycles = []

		# Loop over the cycles
		for cycle_num in range(num_cycles):
			try:
				cycle_start_time = cycle_num * ac_period_s
				cycle_end_time = (cycle_num + 1) * ac_period_s
				subset = data[
					(data[t] >= cycle_start_time) & 
					(data[t] < cycle_end_time)
				].copy().reset_index()

				# Identify zero crossings in smoothed charge
				zero_crossings = np.where(
					np.diff(np.sign(subset['charge_smooth_C'])) != 0
				)[0]

				# Calculate U_delta_pos_V and U_delta_neg_V
				if len(zero_crossings) > 0:
					voltage_just_before = subset.iloc[zero_crossings][U]
					voltage_just_after = subset.iloc[zero_crossings + 1][U]

					voltages_around_crossing = pd.concat(
						[voltage_just_before, voltage_just_after],
						axis=1
					)
					mean_voltages = voltages_around_crossing.mean(axis=1)

					U_delta_pos_V = mean_voltages[mean_voltages > 0].mean()
					U_delta_neg_V = mean_voltages[mean_voltages < 0].mean()
				else:
					U_delta_pos_V = np.nan
					U_delta_neg_V = np.nan

				# Calculate the peak-to-peak voltage
				U_max_pos_V = subset[U].max()
				U_max_neg_V = subset[U].min()
				U_pp_V = np.abs(U_max_pos_V) + np.abs(U_max_neg_V)
				
				#    B---------------A
				#   /               /
				#  /               /
				# C---------------D
				# Calculate indices for A and C and subset data
				# By determining most extreme values of U and Q
				U_max_index = subset[U].idxmax()
				U_min_index = subset[U].idxmin()
				Q_max_index = subset[Q].idxmax()
				Q_min_index = subset[Q].idxmin()

				if Q_max_index < U_min_index:
					subset_A_C = subset.iloc[Q_max_index:U_min_index]
				else:
					subset_A_C = pd.concat(
						[subset.iloc[Q_max_index:], subset.iloc[:U_min_index]]
					)
				
				if Q_min_index < U_max_index:
					subset_C_A = subset.iloc[Q_min_index:U_max_index]
				else:
					subset_C_A = pd.concat(
						[subset.iloc[Q_min_index:], subset.iloc[:U_max_index]]
					)
				
				# Calculate the slopes
				subset_A_B = subset_A_C[
					subset_A_C[U] >= (U_max_pos_V / 3)
				]
				C_cell_neg_F, C_cell_neg_intrcpt_C = np.polyfit(
					subset_A_B[U],
					subset_A_B[Q],
					1
				)

				subset_B_C = subset_A_C[
					subset_A_C[Q] < 0
				]
				C_diel_eff_neg_F, C_diel_eff_neg_intrcpt_C = np.polyfit(
					subset_B_C[U],
					subset_B_C[Q],
					1
				)

				subset_C_D = subset_C_A[
					subset_C_A[U] <= (U_max_neg_V / 3)
				]
				C_cell_pos_F, C_cell_pos_intrcpt_C = np.polyfit(
					subset_C_D[U],
					subset_C_D[Q],
					1
				)

				subset_D_A = subset_C_A[
					subset_C_A[Q] > 0
				]
				C_diel_eff_pos_F, C_diel_eff_pos_intrcpt_C = np.polyfit(
					subset_D_A[U],
					subset_D_A[Q],
					1
				)

				# Calculate slope averages
				C_cell_avg_F = (C_cell_pos_F + C_cell_neg_F) / 2
				C_diel_eff_avg_F = (C_diel_eff_pos_F + C_diel_eff_neg_F) / 2

				# Calculate the alpha and beta parameters
				alpha_pos = (
					(C_diel_F - C_diel_eff_pos_F) /
					(C_diel_F - C_cell_pos_F)
				)
				alpha_neg = (
					(C_diel_F - C_diel_eff_neg_F) /
					(C_diel_F - C_cell_neg_F)
				)
				alpha_avg = (alpha_pos + alpha_neg) / 2

				beta_pos = (
					(C_diel_eff_pos_F - C_cell_pos_F) /
					(C_diel_F - C_cell_pos_F)
				)
				beta_neg = (
					(C_diel_eff_neg_F - C_cell_neg_F) /
					(C_diel_F - C_cell_neg_F)
				)
				beta_avg = (beta_pos + beta_neg) / 2

				# Calculate burning voltages
				U_burning_pos_V = (
					(1 - C_cell_pos_F / C_diel_F) /
					(1 - C_cell_pos_F / C_diel_eff_pos_F) *
					U_delta_pos_V
				)
				U_burning_neg_V = (
					(1 - C_cell_neg_F / C_diel_F) /
					(1 - C_cell_neg_F / C_diel_eff_neg_F) *
					U_delta_neg_V
				)
				U_burning_avg_V = (U_burning_pos_V + np.abs(U_burning_neg_V)) / 2
	
				# Calculate breakdown voltages
				U_breakdown_pos_V = 1 / (1 + C_gas_F / C_diel_F) * U_delta_pos_V
				U_breakdown_neg_V = 1 / (1 + C_gas_F / C_diel_F) * U_delta_neg_V
				U_breakdown_avg_V = (
					U_breakdown_pos_V + np.abs(U_breakdown_neg_V)
				) / 2
	
				# Calculate Q_null_C
				Q_null_C = C_cell_neg_intrcpt_C - C_cell_pos_intrcpt_C

				# Calculate conductively transferred charge
				Q_delta_dis_pos_C = Q_null_C / (1 - C_cell_pos_F / C_diel_F)
				Q_delta_dis_neg_C = Q_null_C / (1 - C_cell_neg_F / C_diel_F)
				Q_delta_dis_avg_C = (Q_delta_dis_pos_C + Q_delta_dis_neg_C) / 2

				# Calculate plasma power (W)
				power_plasma_W = lissajous_area(
					subset[U], subset[Q]
				) * ac_freq_Hz

				# Calculate plasma power using the product of U and I
				power_plasma_UIprod_W = np.mean(subset[U] * subset[Ir])

				# Calculate source power (W)
				power_source_W = np.mean(subset[U] * subset[Is])

				# Calculate the RMS current in the reactor
				current_rms_reactor_A = np.sqrt(np.mean(np.square(subset[Ir])))

				# Calculate the RMS current in the source
				current_rms_source_A = np.sqrt(np.mean(np.square(subset[Is])))

				# Calculate plasma current
				subset['current_plasma_A'] = (
					1 / (1 - C_cell_avg_F / C_diel_F)
					) * (
					subset[Ir] - C_cell_avg_F * subset['dV_dt']
				)

				current_rms_plasma_A = np.sqrt(
					np.mean(np.square(subset['current_plasma_A']))
				)

				# Find peaks in the plasma current
				peak_distance = int(50e-9 / t_per_row)
				peak_width = int(15e-9 / t_per_row)
				peaks, _ = find_peaks(
					subset['current_plasma_A'],
					height=discharge_threshold_A,
					width=peak_width,
					distance=peak_distance,
					prominence=discharge_threshold_A
				)

				# Count the number of peaks
				num_udisch = len(peaks)

				# Write cycle results to results_cycles
				results_cycles.append(
					{
						'project_name': project_name,
						'psdata_file_name': psdata_file_name,
						'cycle': cycle_num + 1,
						'measurement_date': date,
						'measurement_number': measurement_number,
						'material_supplier': material_supplier,
						'material_name': material_name,
						'reaction_type': reaction_type,
						'wattage_const': wattage_const,
						'residence_time_s': residence_time_s,
						'plasma_state': plasma_state,
						'power_plasma_W': power_plasma_W,
						'power_plasma_UIprod_W': power_plasma_UIprod_W,
						'power_source_W': power_source_W,
						'U_pp_V': U_pp_V,
						'current_rms_reactor_A': current_rms_reactor_A,
						'current_rms_plasma_A': current_rms_plasma_A,
						'current_rms_source_A': current_rms_source_A,
						'U_burning_neg_V': U_burning_neg_V,
						'U_burning_pos_V': U_burning_pos_V,
						'U_burning_avg_V': U_burning_avg_V,
						'U_breakdown_neg_V': U_breakdown_neg_V,
						'U_breakdown_pos_V': U_breakdown_pos_V,
						'U_breakdown_avg_V': U_breakdown_avg_V,
						'Q_delta_dis_pos_C': Q_delta_dis_pos_C,
						'Q_delta_dis_neg_C': Q_delta_dis_neg_C,
						'Q_delta_dis_avg_C': Q_delta_dis_avg_C,
						'avg_num_udisch_per_cycle': num_udisch,
						'C_cell_neg_F': C_cell_neg_F,
						'C_cell_neg_intrcpt_C': C_cell_neg_intrcpt_C,
						'C_cell_pos_F': C_cell_pos_F,
						'C_cell_pos_intrcpt_C': C_cell_pos_intrcpt_C,
						'C_cell_avg_F': C_cell_avg_F,
						'C_diel_eff_neg_F': C_diel_eff_neg_F,
						'C_diel_eff_neg_intrcpt_C': C_diel_eff_neg_intrcpt_C,
						'C_diel_eff_pos_F': C_diel_eff_pos_F,
						'C_diel_eff_pos_intrcpt_C': C_diel_eff_pos_intrcpt_C,
						'C_diel_eff_avg_F': C_diel_eff_avg_F,
						'alpha_neg': alpha_neg,
						'alpha_pos': alpha_pos,
						'alpha_avg': alpha_avg,
						'beta_neg': beta_neg,
						'beta_pos': beta_pos,
						'beta_avg': beta_avg
					}
				)
			except Exception as e:
				print(f'Error processing cycle {cycle_num + 1} of {num_cycles}...')
				print(e)
				continue
		
		# Write cycle results to results_psdata_files
		results_psdata_files.extend(results_cycles)

	# Create a DataFrame from the results
	results_df = pd.DataFrame(
		results_psdata_files,
		columns=result_column_names
	)

	return data_df, results_df

Step 2: `remove_outliers`

Define remove_outliers function that removes outliers from the generated DataFrame in the previous step

In [ ]:
def remove_outliers(results_df):
	try:
		columns_to_predict_outliers = [
			'power_plasma_W',
			'U_pp_V',
			'current_rms_plasma_A',
			'U_burning_avg_V',
			'U_breakdown_avg_V',
			'Q_delta_dis_avg_C',
			'avg_num_udisch_per_cycle',
			'C_cell_avg_F',
			'C_diel_eff_avg_F'
		]

		# Scale the data
		scaled_data = MaxAbsScaler().fit_transform(
			results_df[columns_to_predict_outliers]
		)

		# Determine outliers
		clf = EllipticEnvelope(
			random_state=42,
			support_fraction=0.95,
			contamination=0.25
		)
		outlier_predictions = clf.fit_predict(scaled_data)

		# Add a new column 'outlier' to the data
		# with the outlier predictions
		results_df['outlier'] = outlier_predictions

		# Filter the outliers
		no_outliers = results_df[
			results_df['outlier'] == 1
		].drop(columns=['outlier'])

		print(f'Found {results_df.shape[0] - no_outliers.shape[0]} outliers.')

		# Return the no_outliers DataFrame
		return no_outliers

	except Exception as e:
		print('An error occurred while processing for outlier detection in:'
		f'{results_df['psdata_file_name'].iloc[0]}. Error message: {str(e)}')

Step 3: `lissajous_generation`

Define lissajous_generation function that generates lissajous figures for the given DataFrame

In [ ]:
def lissajous_figure_generation(
	data_df,
	results_no_outliers_df,
	folder_name,
	row_skip=100,
	time_column='time_s',
	voltage_column='voltage_V',
	charge_column='charge_C',
	voltage_charge_deriv_column='dQ_dV'
):
	try:
		columns_to_keep = [
			time_column,
			voltage_column,
			charge_column,
			voltage_charge_deriv_column
		]
		# Keep the columns that are needed for the lissajous generation
		data_df = data_df[columns_to_keep]

		# Group by 'time_s' and aggregate
		data_mean = data_df.groupby('time_s').mean().reset_index()

		# Select every row_skip row
		data_subset = data_mean.iloc[::row_skip].reset_index()
		
		# Extract and average slopes and intercepts
		C_cell_pos_F = results_no_outliers_df['C_cell_pos_F'].mean()
		C_cell_neg_F = results_no_outliers_df['C_cell_neg_F'].mean()
		C_cell_pos_intrcpt_C = results_no_outliers_df[
			'C_cell_pos_intrcpt_C'
		].mean()
		C_cell_neg_intrcpt_C = results_no_outliers_df[
			'C_cell_neg_intrcpt_C'
		].mean()

		C_diel_eff_pos_F = results_no_outliers_df['C_diel_eff_pos_F'].mean()
		C_diel_eff_neg_F = results_no_outliers_df['C_diel_eff_neg_F'].mean()
		C_diel_eff_pos_intrcpt_C = results_no_outliers_df[
			'C_diel_eff_pos_intrcpt_C'
		].mean()
		C_diel_eff_neg_intrcpt_C = results_no_outliers_df[
			'C_diel_eff_neg_intrcpt_C'
		].mean()

		# calculate lines
		x = np.arange(-1.2e4, 1.2e4, 1e3)
		y_cell_pos = C_cell_pos_F * x + C_cell_pos_intrcpt_C
		y_cell_neg = C_cell_neg_F * x + C_cell_neg_intrcpt_C
		y_diel_eff_pos = C_diel_eff_pos_F * x + C_diel_eff_pos_intrcpt_C
		y_diel_eff_neg = C_diel_eff_neg_F * x + C_diel_eff_neg_intrcpt_C

		# Plot and save the lissajous figure and dQ_dV vs time_s figure side by side
		plt.figure(figsize=(14, 6))
		plt.subplot(1, 2, 1)
		plt.plot(
			data_subset['voltage_V'],
			data_subset['charge_C'],
			label='Lissajous figure'
		)
		plt.plot(x, y_cell_pos, label='CD, C_cell_pos')
		plt.plot(x, y_cell_neg, label='AB, C_cell_neg')
		plt.plot(x, y_diel_eff_pos, label='DA, C_diel_eff_pos')
		plt.plot(x, y_diel_eff_neg, label='BC, C_diel_eff_neg')
		plt.axis([-15e3, 15e3, -1e-6, 1e-6])
		plt.xlabel('Voltage (V)')
		plt.ylabel('Charge (C)')
		plt.legend()

		plt.subplot(1, 2, 2)
		plt.plot(
			data_subset['time_s'],
			abs(data_subset['dQ_dV']),
			label='dQ_dV vs time_s'
		)
		plt.xlabel('Time (s)')
		plt.ylabel('dQ_dV (F)')
		plt.axis([0, 1e-3, 0, 200e-12])
		plt.legend()

		plot_filename = (
			folder_name +
			'/' +
			'lissajous_figure.png'
		)

		plt.savefig(plot_filename, dpi=150)
		plt.clf()
		plt.close()
		print(f'Lissajous figure saved as {plot_filename}')
		
	except Exception as e:
			print('An error occurred while averaging the lissajous figures.'
				  f'Error message: {str(e)}')

Step 4: `profile_calculation`

Define 'current_voltage_profile' function that generates the current/voltage/charge/dQ_dV profile for the given DataFrame by grouping the data by 'filename', selecting every 'row_skip'th row, creating a column 'wave_index' that contains the index of the current wavelength. Create a column 'current_sign' that indecates if the current is positive or negative.

In [ ]:
def profile_calculation(
	data_df,
	results_no_outliers_df,
	C_diel,
	row_skip=100,
	ac_freq_Hz=3000
):
	try:
		# Initialize an empty DataFrame to store the subset
		data_subset = pd.DataFrame()

		# Group data by 'psdata_file_name' and iterate over each group
		for group_name, group_data in data_df.groupby('psdata_file_name'):
			# Select every 'row_skip'th row from the group
			subset = group_data.iloc[::row_skip, :].reset_index(drop=True)
			
			# Extract variables from the results_df where
			# the 'psdata_file_name' is equal to the group_name
			alpha = results_no_outliers_df.loc[
				results_no_outliers_df['psdata_file_name'] == group_name, 'alpha_avg'
			].iloc[0]
			
			beta = results_no_outliers_df.loc[
				results_no_outliers_df['psdata_file_name'] == group_name, 'beta_avg'
			].iloc[0]
			
			C_cell = results_no_outliers_df.loc[
				results_no_outliers_df['psdata_file_name'] == group_name, 'C_cell_avg_F'
			].iloc[0]

			# Calculate plasma current
			subset['current_plasma_A'] = (1 / (1 - C_cell / C_diel)) * (
				subset['current_reactor_A'] - C_cell * subset['dV_dt']
			)

			# Calculate displacement current
			subset['current_displacement_A'] = (
				subset['current_reactor_A'] - subset['current_plasma_A']
			)

			# Calculate plasma charge
			subset['charge_plasma_C'] = (
				subset['charge_C'] - C_cell * subset['voltage_V']
			) / (1 - C_cell / C_diel)

			# Calculate gap voltage
			subset['voltage_gap_V'] = (
				1 + (alpha * C_cell) / (beta * C_diel)
			) * subset['voltage_V'] - (1 / (beta * C_diel)) * subset['charge_C']
			
			# Drop the unnecessary column
			subset = subset.drop(
				columns=['dV_dt', 'voltage_smooth_V', 'charge_smooth_C'],
				errors='ignore'
			)
			
			# Append the subset to the data_subset DataFrame
			data_subset = pd.concat([data_subset, subset], ignore_index=True)

		# Create a column 'wave_index' that contains the index of the
		# current wavelength, a full wavelength is 1/3000 seconds.
		# time_s from 0 to <1/3000 is wave_index 0,
		# from 1/3000 to <2/3000 is wave_index 1, etc.,
		# until 5/3000 to <6/3000 is wave_index 5.
		data_subset['wave_index'] = (
			(data_subset['time_s'] * ac_freq_Hz).astype(int) % 6
		)

		# Create 'current_index'
		# Create 'current_sign' indicating for 0 to <1/6000 that
		# the current is positive (1) and for 1/6000 to <2/6000 that
		# the current is negative (-1) and so on until <12/6000 (-1)
		data_subset['current_index'] = (
			(data_subset['time_s'] * (ac_freq_Hz * 2)).astype(int) % 12
		)
		
		data_subset['current_sign'] = (-1) ** data_subset['current_index']

		print('Profile calculation completed.')

		return data_subset

	except Exception as e:
		print(f'An error occurred: {e}')
		return None

Step 5: Run the functions sequentially

In [ ]:
# Walk through each directory in the folder tree and do the calculations
for folder_name, _, filenames in os.walk(top_folder):
	try:
		# Check if 'lissajous_data.parquet' is not in the directory
		if 'lissajous_data.parquet' not in filenames:
			continue

		else:
			print(f'Processing folder: {folder_name}')

			# Delete the .png, .csv, and .parquet result files
			for file in os.listdir(folder_name):
				if (
					(file.endswith('.csv') or file.endswith('.png') or file.endswith('.parquet')) and
					file != 'lissajous_data.parquet'
				):
					os.remove(os.path.join(folder_name, file))

			# Read the lissajous_data.parquet file
			data_df = pd.read_parquet(
				os.path.join(folder_name, 'lissajous_data.parquet')
			)

			data_df, results_df = calculate_results(
				data_df,
				result_column_names=column_names_new,
				ac_freq_Hz=3000,
				C_diel_F=204e-12,
				C_gas_F=7.4e-12,
				discharge_threshold_A=7.5e-3,
				t='time_s',
				Q='charge_C',
				U='voltage_V',
				Ir='current_reactor_A',
				Is='current_source_A',
				smoothing_window_size=15625
			)

			results_df.to_parquet(
				os.path.join(
					folder_name,
					'lissajous_calculations.parquet'
				),
				index=False,
				compression='gzip'
			)

			# Remove outliers from the results
			results_no_outliers_df = remove_outliers(
				results_df=results_df
			)

			# Write the results without outliers to a .parquet file
			results_no_outliers_df.to_parquet(
				os.path.join(
					folder_name,
					'lissajous_calculations_no_outliers.parquet'
				),
				index=False,
				compression='gzip'
			)

			# Select only numeric columns for averaging
			numeric_columns = results_no_outliers_df.select_dtypes(
				include=[float, int]
			).columns

			# Group by 'psdata_file_name' and calculate the mean for numeric columns
			results_no_outliers_avg_df = results_no_outliers_df.groupby(
				'psdata_file_name'
			)[numeric_columns].mean().reset_index()

			# Write the averaged results to a .csv file
			results_no_outliers_avg_df.to_csv(
				os.path.join(
					folder_name,
					'lissajous_calculations_no_outliers_avg.csv'
				),
				index=False
			)
			
			# Generate mean lissajous figure
			lissajous_figure_generation(
				data_df=data_df,
				results_no_outliers_df=results_no_outliers_df,
				folder_name=folder_name,
				time_column='time_s',
				voltage_column='voltage_V',
				charge_column='charge_C',
				row_skip=100
			)
			
			# Generate current/voltage profile
			profile_df = profile_calculation(
				data_df=data_df,
				results_no_outliers_df=results_no_outliers_df,
				C_diel=204e-12,
				row_skip=100,
				ac_freq_Hz=3000
			)
			
			# Write the current/voltage profile to a .parquet file
			profile_df.to_parquet(
				os.path.join(
					folder_name,
					'profile_data.parquet'
				),
				index=False,
				compression='gzip'
			)
			
			# Free up memory
			gc.collect()
	except Exception as e:
		print(f'An error occurred: {e}')
		continue

In [ ]:
# Clear variables
gc.collect()
%reset -f